In [41]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [19]:
df = pd.read_csv('/ceph/MethDev/pbio/kay/data/annotated_filtered_col.CG_2.fast.tsv', sep='\t')      # or read_parquet / feather …
df


,cluster,chr,start,end,score,c,t,n,flag_euc_gene,flag_het_gene,flag_euc_TE,flag_het_TE,score_masked
0,0,1,101,200,0.8951,350,41,6,False,False,False,False,0.8951
1,0,1,301,400,0.5487,62,51,2,False,False,False,False,0.5487
2,0,1,401,500,0.8246,47,10,1,False,False,False,False,0.8246
3,0,1,501,600,0.7206,98,38,3,False,False,False,False,0.7206
4,0,1,601,700,0.8982,203,23,6,False,False,False,False,0.8982
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5925984,16,5,26974801,26974900,0.8462,22,4,7,False,False,False,False,0.8462
5925985,16,5,26974901,26975000,0.7500,3,1,2,False,False,False,False,NaN
5925986,16,5,26975101,26975200,1.0000,4,0,4,False,False,False,False,NaN
5925987,16,5,26975201,26975300,1.0000,8,0,8,False,False,False,False,1.0000


In [20]:
df[(df['start']==21328401) & (df['chr']==1)]	

,cluster,chr,start,end,score,c,t,n,flag_euc_gene,flag_het_gene,flag_euc_TE,flag_het_TE,score_masked
61558,0,1,21328401,21328500,0.7649,257,79,8,True,False,False,False,0.7649
415910,1,1,21328401,21328500,0.6847,278,128,8,True,False,False,False,0.6847
770164,2,1,21328401,21328500,0.8054,120,29,8,True,False,False,False,0.8054
1124039,3,1,21328401,21328500,0.8770,107,15,8,True,False,False,False,0.8770
1477777,4,1,21328401,21328500,0.7664,105,32,8,True,False,False,False,0.7664
1831620,5,1,21328401,21328500,0.8846,69,9,8,True,False,False,False,0.8846
2184900,6,1,21328401,21328500,0.6887,73,33,8,True,False,False,False,0.6887
2538727,7,1,21328401,21328500,0.8305,98,20,8,True,False,False,False,0.8305
2892070,8,1,21328401,21328500,0.7436,58,20,8,True,False,False,False,0.7436
3243535,9,1,21328401,21328500,0.5526,21,17,8,True,False,False,False,0.5526


chr, start, end — the genomic window (your coords are 1-based inclusive).

category — which baseline this window was judged against (e.g., euc_gene). This matters because each category has its own typical methylation level across clusters.

X2 — the Pearson chi-square statistic for this window, comparing observed methylated counts in each cluster to what we’d expect given the category/feature’s cluster offsets (i.e., the usual pattern for that category). Bigger = more deviation from the baseline pattern.

df — degrees of freedom = (#clusters with coverage in this window) − 1. You’ve got 16 here, which means 17 clusters contributed (e.g., cluster IDs 0–16). If you truly have 16 clusters, two possibilities: either indexing is 0–16 (17 total) or one extra cluster slipped in—worth a quick check.

phi — the overdispersion estimate for the whole category (constant within a category block). phi ≈ 1.54 means counts vary ~1.54× more than simple binomial would predict; we divide X2 by phi before computing the p-value so we don’t overcall.

pval — p-value from a χ²(df) test using X2 / phi. This tests “does this window’s across-cluster profile deviate from the category trend?”

qval — BH-FDR adjusted p-value within this category (so you can compare windows fairly inside e.g. euc_gene).

delta_max — EB-shrunken effect size for this window: the max difference in methylation fraction between any two clusters after shrinking low-coverage estimates toward the category mean. It’s on the same 0–1 scale as methylation fraction (so 0.30 ≈ 30 percentage points).

hi_cluster / lo_cluster — which clusters achieve that delta_max contrast (highest vs lowest shrunken methylation for this window).

How to interpret a row (example) chr1:11201-11300 (euc_gene):

X2=77.02, df=16, phi=1.5368 → strong deviation from the expected pattern for euc_gene windows.

pval=2.2e-5, qval=4.18e-4 → significant after FDR within euc_gene.

delta_max=0.443, hi_cluster=15, lo_cluster=0 → after EB shrinkage, cluster 15 is ~44 percentage points more methylated than cluster 0 at this window. That’s a big effect.

A row with df=14 (e.g., 15260301–15260400) just means 2 clusters had zero coverage and were excluded from the test for that window.

In [27]:
#not including non-overlapping regions, excluded globally low methylated rows
dmw = pd.read_pickle("/ceph/MethDev/pbio/kay/data/dmw_robust.pkl")
dmw

,chr,start,end,X2,df,delta_max,hi_cluster,lo_cluster,delta_max_trim,top_cluster,...,X2_loco,df_loco,phi,pval,p_loco,qval,neighbor_support,dominance_blocked,reason,category
0,1,5401,5500,13.760077,15,0.141937,11,1,0.102152,6,...,10.940558,14.0,1.536786,0.879921,0.930022,1.000000,0,True,ns_qval,euc_gene
1,1,7001,7100,55.949639,16,0.298853,14,9,0.220058,9,...,44.432533,15.0,1.536786,0.002540,0.016506,0.023845,0,False,no_neighbor_support,euc_gene
2,1,11201,11300,77.019910,16,0.443047,15,0,0.350130,12,...,20.305446,15.0,1.536786,0.000022,0.585855,0.000418,0,True,low_cov_hi_lo_and_loco_ns,euc_gene
3,1,23901,24000,19.171355,15,0.243401,3,4,0.156362,4,...,14.295747,14.0,1.536786,0.642780,0.811207,1.000000,1,True,ns_qval,euc_gene
4,1,24001,24100,21.045295,15,0.197810,14,2,0.150443,2,...,15.004330,14.0,1.536786,0.548825,0.779265,0.984130,2,True,ns_qval,euc_gene
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
304685,5,15299801,15299900,18.546799,16,0.197922,12,1,0.136665,16,...,14.141530,15.0,1.544736,0.743535,0.869293,1.000000,1,True,ns_qval,het_te
304686,5,15299901,15300000,13.166474,16,0.189137,5,16,0.077046,3,...,9.454273,15.0,1.544736,0.931736,0.977655,1.000000,0,True,ns_qval,het_te
304687,5,15300001,15300100,14.476215,16,0.200705,7,1,0.129774,8,...,7.914641,15.0,1.544736,0.897338,0.991027,1.000000,0,True,ns_qval,het_te
304688,5,15300201,15300300,16.898293,16,0.133333,13,8,0.110645,8,...,10.744740,15.0,1.544736,0.813218,0.958867,1.000000,0,True,ns_qval,het_te


In [28]:
dmw = dmw.drop(["X2_loco", "df_loco", "p_loco", "dominance_blocked", "reason"], axis = 1)
dmw

,chr,start,end,X2,df,delta_max,hi_cluster,lo_cluster,delta_max_trim,top_cluster,top_cov,m_hi,m_lo,phi,pval,qval,neighbor_support,category
0,1,5401,5500,13.760077,15,0.141937,11,1,0.102152,6,91,18.0,449.0,1.536786,0.879921,1.000000,0,euc_gene
1,1,7001,7100,55.949639,16,0.298853,14,9,0.220058,9,20,15.0,20.0,1.536786,0.002540,0.023845,0,euc_gene
2,1,11201,11300,77.019910,16,0.443047,15,0,0.350130,12,8,2.0,113.0,1.536786,0.000022,0.000418,0,euc_gene
3,1,23901,24000,19.171355,15,0.243401,3,4,0.156362,4,63,36.0,63.0,1.536786,0.642780,1.000000,1,euc_gene
4,1,24001,24100,21.045295,15,0.197810,14,2,0.150443,2,54,6.0,54.0,1.536786,0.548825,0.984130,2,euc_gene
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
304685,5,15299801,15299900,18.546799,16,0.197922,12,1,0.136665,16,11,11.0,187.0,1.544736,0.743535,1.000000,1,het_te
304686,5,15299901,15300000,13.166474,16,0.189137,5,16,0.077046,3,84,58.0,12.0,1.544736,0.931736,1.000000,0,het_te
304687,5,15300001,15300100,14.476215,16,0.200705,7,1,0.129774,8,12,26.0,137.0,1.544736,0.897338,1.000000,0,het_te
304688,5,15300201,15300300,16.898293,16,0.133333,13,8,0.110645,8,10,10.0,10.0,1.544736,0.813218,1.000000,0,het_te


In [29]:
dmw[dmw['start']==21328401]

,chr,start,end,X2,df,delta_max,hi_cluster,lo_cluster,delta_max_trim,top_cluster,top_cov,m_hi,m_lo,phi,pval,qval,neighbor_support,category
24606,1,21328401,21328500,81.411348,16,0.416370,3,10,0.283860,10,15,122.0,15.0,1.536786,0.000008,0.000164,1,euc_gene
80514,3,21328401,21328500,50.511393,16,0.220475,8,9,0.165404,1,483,60.0,61.0,1.536786,0.007693,0.058166,2,euc_gene
126605,5,21328401,21328500,7.060000,16,0.174067,11,4,0.135225,12,14,5.0,93.0,1.536786,0.997431,1.000000,0,euc_gene


In [17]:
def select_dmws_strict(
    dmw,
    # base gate
    qval=0.02,
    delta_max=0.15,
    
    # main path
    neighbor_min_main=1,
    m_floor_main=10,
    
    # rescue path
    rescue_delta=0.255,
    rescue_m_hi=7,
    rescue_m_lo=50,
    
    # consensus path
    neighbor_min_consensus=2,
    m_floor_consensus=20,
):
    dmw = dmw.copy()

    base = (
        (dmw['qval'] <= qval) &
        (dmw['delta_max'] >= delta_max)
    )

    main = (
        base &
        (dmw['neighbor_support'] >= neighbor_min_main) &
        (dmw['m_hi'] >= m_floor_main) &
        (dmw['m_lo'] >= m_floor_main)
    )

    rescue = (
        base &
        (dmw['delta_max_trim'] >= rescue_delta) &
        (dmw['m_hi'] >= rescue_m_hi) &
        (dmw['m_lo'] >= rescue_m_lo)
    )

    consensus = (
        base &
        (dmw['neighbor_support'] >= neighbor_min_consensus) &
        (dmw['m_hi'] >= m_floor_consensus) &
        (dmw['m_lo'] >= m_floor_consensus)
    )

    call_mask = main | rescue | consensus

    # Build a full-length reason array aligned to dmw.index
    call_reason = np.full(len(dmw), 'none', dtype=object)
    # set priority (choose the precedence you want)
    call_reason[rescue.to_numpy()]   = 'rescue_isolated'
    call_reason[consensus.to_numpy()] = 'consensus_neighbors'
    call_reason[main.to_numpy()]     = 'main'  # <- highest precedence

    dmw['call_reason'] = call_reason

    calls = dmw.loc[call_mask].copy()

    # # Optional: rank within calls
    # min_cov = np.minimum(calls['m_hi'], calls['m_lo'])
    # calls['priority'] = (
    #     (-np.log10(np.clip(calls['qval'], 1e-300, 1))) *
    #     np.maximum(calls['delta_max_trim'], 0.01) *
    #     (min_cov / (min_cov + 20)) *
    #     (1 + calls['neighbor_support'])
    # )

    return calls#.sort_values(['call_reason', 'priority'], ascending=[True, False])


In [18]:
calls_strict = select_dmws_strict(dmw)
calls_strict

,chr,start,end,X2,df,delta_max,hi_cluster,lo_cluster,delta_max_trim,top_cluster,top_cov,m_hi,m_lo,phi,pval,qval,neighbor_support,category,call_reason
7,1,24301,24400,63.303077,16,0.323758,0,15,0.235239,15,16,511.0,16.0,1.536786,5.201946e-04,6.429773e-03,1,euc_gene,main
85,1,77801,77900,283.205748,16,0.509852,16,0,0.258787,16,7,7.0,355.0,1.536786,0.000000e+00,0.000000e+00,0,euc_gene,rescue_isolated
110,1,99801,99900,93.766555,16,0.430181,14,1,0.292976,14,10,10.0,433.0,1.536786,3.527498e-07,1.039362e-05,0,euc_gene,rescue_isolated
125,1,117201,117300,209.579299,16,0.469321,16,0,0.315957,16,8,8.0,640.0,1.536786,0.000000e+00,0.000000e+00,1,euc_gene,rescue_isolated
129,1,121701,121800,121.351245,16,0.345870,4,16,0.321142,9,38,222.0,13.0,1.536786,2.556039e-10,1.349449e-08,1,euc_gene,main
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
304610,5,15242701,15242800,74.835948,16,0.249490,3,12,0.184068,12,59,160.0,59.0,1.544736,4.041713e-05,2.069924e-03,1,het_te,main
304611,5,15242801,15242900,77.878127,16,0.302944,13,12,0.154973,12,62,59.0,62.0,1.544736,1.968215e-05,1.171208e-03,2,het_te,main
304613,5,15243001,15243100,86.855885,16,0.244768,8,9,0.176046,12,66,45.0,55.0,1.544736,2.232159e-06,1.974125e-04,2,het_te,main
304627,5,15259701,15259800,70.739222,16,0.413612,8,6,0.340276,6,56,25.0,56.0,1.544736,1.047749e-04,4.410364e-03,1,het_te,main


In [31]:
def select_dmws_loose(
    dmw,
    # base gate
    qval=0.05,
    delta_max=0.15,
    
    # main path
    neighbor_min_main=1,
    m_floor_main=10,
    
    # rescue path
    rescue_delta=0.255,
    rescue_m_hi=7,
    rescue_m_lo=50,
    
    # consensus path
    neighbor_min_consensus=2,
    m_floor_consensus=20,
):
    dmw = dmw.copy()

    base = (
        (dmw['qval'] <= qval) &
        (dmw['delta_max'] >= delta_max)
    )

    main = (
        base &
        (dmw['neighbor_support'] >= neighbor_min_main) &
        (dmw['m_hi'] >= m_floor_main) &
        (dmw['m_lo'] >= m_floor_main)
    )

    rescue = (
        base &
        (dmw['delta_max_trim'] >= rescue_delta) &
        (dmw['m_hi'] >= rescue_m_hi) &
        (dmw['m_lo'] >= rescue_m_lo)
    )

    consensus = (
        base &
        (dmw['neighbor_support'] >= neighbor_min_consensus) &
        (dmw['m_hi'] >= m_floor_consensus) &
        (dmw['m_lo'] >= m_floor_consensus)
    )

    call_mask = main | rescue | consensus

    # Build a full-length reason array aligned to dmw.index
    call_reason = np.full(len(dmw), 'none', dtype=object)
    # set priority (choose the precedence you want)
    call_reason[rescue.to_numpy()]   = 'rescue_isolated'
    call_reason[consensus.to_numpy()] = 'consensus_neighbors'
    call_reason[main.to_numpy()]     = 'main'  # <- highest precedence

    dmw['call_reason'] = call_reason

    calls = dmw.loc[call_mask].copy()

    # # Optional: rank within calls
    # min_cov = np.minimum(calls['m_hi'], calls['m_lo'])
    # calls['priority'] = (
    #     (-np.log10(np.clip(calls['qval'], 1e-300, 1))) *
    #     np.maximum(calls['delta_max_trim'], 0.01) *
    #     (min_cov / (min_cov + 20)) *
    #     (1 + calls['neighbor_support'])
    # )

    return calls#.sort_values(['call_reason', 'priority'], ascending=[True, False])


In [32]:
calls_loose = select_dmws_loose(dmw)
calls_loose

,chr,start,end,X2,df,delta_max,hi_cluster,lo_cluster,delta_max_trim,top_cluster,top_cov,m_hi,m_lo,phi,pval,qval,neighbor_support,category,call_reason
7,1,24301,24400,63.303077,16,0.323758,0,15,0.235239,15,16,511.0,16.0,1.536786,5.201946e-04,0.006430,1,euc_gene,main
27,1,26601,26700,54.791427,16,0.340057,0,12,0.204524,12,17,241.0,17.0,1.536786,3.231771e-03,0.029005,2,euc_gene,main
85,1,77801,77900,283.205748,16,0.509852,16,0,0.258787,16,7,7.0,355.0,1.536786,0.000000e+00,0.000000,0,euc_gene,rescue_isolated
110,1,99801,99900,93.766555,16,0.430181,14,1,0.292976,14,10,10.0,433.0,1.536786,3.527498e-07,0.000010,0,euc_gene,rescue_isolated
125,1,117201,117300,209.579299,16,0.469321,16,0,0.315957,16,8,8.0,640.0,1.536786,0.000000e+00,0.000000,1,euc_gene,rescue_isolated
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
304612,5,15242901,15243000,56.581628,16,0.234919,3,16,0.158731,12,71,138.0,27.0,1.544736,2.364557e-03,0.044639,2,het_te,main
304613,5,15243001,15243100,86.855885,16,0.244768,8,9,0.176046,12,66,45.0,55.0,1.544736,2.232159e-06,0.000197,2,het_te,main
304627,5,15259701,15259800,70.739222,16,0.413612,8,6,0.340276,6,56,25.0,56.0,1.544736,1.047749e-04,0.004410,1,het_te,main
304628,5,15259801,15259900,67.105317,16,0.465839,5,4,0.305060,4,83,32.0,83.0,1.544736,2.397276e-04,0.008304,2,het_te,main


In [37]:
def select_dmws_strict_base(
    dmw,
    # base gate
    qval=0.01,
    delta_max=0.15,
    
    # main path
    neighbor_min_main=1,
    m_floor_main=10,
    
    # rescue path
    rescue_delta=0.255,
    rescue_m_hi=7,
    rescue_m_lo=50,
    
    # consensus path
    neighbor_min_consensus=2,
    m_floor_consensus=20,
):
    dmw = dmw.copy()

    base = (
        (dmw['qval'] <= qval) &
        (dmw['delta_max'] >= delta_max)
    )


    call_mask = base

    # # Build a full-length reason array aligned to dmw.index
    # call_reason = np.full(len(dmw), 'none', dtype=object)
    # # set priority (choose the precedence you want)
    # call_reason[rescue.to_numpy()]   = 'rescue_isolated'
    # call_reason[consensus.to_numpy()] = 'consensus_neighbors'
    # call_reason[main.to_numpy()]     = 'main'  # <- highest precedence

    # dmw['call_reason'] = call_reason

    calls = dmw.loc[call_mask].copy()

    # # Optional: rank within calls
    # min_cov = np.minimum(calls['m_hi'], calls['m_lo'])
    # calls['priority'] = (
    #     (-np.log10(np.clip(calls['qval'], 1e-300, 1))) *
    #     np.maximum(calls['delta_max_trim'], 0.01) *
    #     (min_cov / (min_cov + 20)) *
    #     (1 + calls['neighbor_support'])
    # )

    return calls#.sort_values(['call_reason', 'priority'], ascending=[True, False])


In [38]:
calls_strict_base = select_dmws_strict_base(dmw)
calls_strict_base

,chr,start,end,X2,df,delta_max,hi_cluster,lo_cluster,delta_max_trim,top_cluster,top_cov,m_hi,m_lo,phi,pval,qval,neighbor_support,category
2,1,11201,11300,77.019910,16,0.443047,15,0,0.350130,12,8,2.0,113.0,1.536786,2.195709e-05,0.000418,0,euc_gene
7,1,24301,24400,63.303077,16,0.323758,0,15,0.235239,15,16,511.0,16.0,1.536786,5.201946e-04,0.006430,1,euc_gene
20,1,25801,25900,68.401063,16,0.283734,0,16,0.233424,9,61,373.0,8.0,1.536786,1.649916e-04,0.002425,2,euc_gene
72,1,40001,40100,80.305042,16,0.399639,8,0,0.333611,16,5,3.0,88.0,1.536786,9.962041e-06,0.000209,0,euc_gene
75,1,48101,48200,93.444025,16,0.466299,16,0,0.328892,8,9,1.0,110.0,1.536786,3.827983e-07,0.000011,0,euc_gene
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
304610,5,15242701,15242800,74.835948,16,0.249490,3,12,0.184068,12,59,160.0,59.0,1.544736,4.041713e-05,0.002070,1,het_te
304611,5,15242801,15242900,77.878127,16,0.302944,13,12,0.154973,12,62,59.0,62.0,1.544736,1.968215e-05,0.001171,2,het_te
304613,5,15243001,15243100,86.855885,16,0.244768,8,9,0.176046,12,66,45.0,55.0,1.544736,2.232159e-06,0.000197,2,het_te
304627,5,15259701,15259800,70.739222,16,0.413612,8,6,0.340276,6,56,25.0,56.0,1.544736,1.047749e-04,0.004410,1,het_te


In [45]:
calls_strict_base['X2'].describe()

count    18693.000000
mean       106.715347
std         92.850526
min         51.488440
25%         70.571088
50%         83.300656
75%        110.130668
max       5218.178603
Name: X2, dtype: float64

In [47]:
def select_dmws_loose_base(
    dmw,
    # base gate
    qval=0.05,
    delta_max=0.15,
    
    # main path
    neighbor_min_main=1,
    m_floor_main=10,
    
    # rescue path
    rescue_delta=0.255,
    rescue_m_hi=7,
    rescue_m_lo=50,
    
    # consensus path
    neighbor_min_consensus=2,
    m_floor_consensus=20,
):
    dmw = dmw.copy()

    base = (
        (dmw['qval'] <= qval) &
        (dmw['delta_max'] >= delta_max)
    )


    call_mask = base

    # # Build a full-length reason array aligned to dmw.index
    # call_reason = np.full(len(dmw), 'none', dtype=object)
    # # set priority (choose the precedence you want)
    # call_reason[rescue.to_numpy()]   = 'rescue_isolated'
    # call_reason[consensus.to_numpy()] = 'consensus_neighbors'
    # call_reason[main.to_numpy()]     = 'main'  # <- highest precedence

    # dmw['call_reason'] = call_reason

    calls = dmw.loc[call_mask].copy()

    # # Optional: rank within calls
    # min_cov = np.minimum(calls['m_hi'], calls['m_lo'])
    # calls['priority'] = (
    #     (-np.log10(np.clip(calls['qval'], 1e-300, 1))) *
    #     np.maximum(calls['delta_max_trim'], 0.01) *
    #     (min_cov / (min_cov + 20)) *
    #     (1 + calls['neighbor_support'])
    # )

    return calls#.sort_values(['call_reason', 'priority'], ascending=[True, False])


In [48]:
calls_loose_base = select_dmws_loose_base(dmw)
calls_loose_base

,chr,start,end,X2,df,delta_max,hi_cluster,lo_cluster,delta_max_trim,top_cluster,top_cov,m_hi,m_lo,phi,pval,qval,neighbor_support,category
1,1,7001,7100,55.949639,16,0.298853,14,9,0.220058,9,20,15.0,20.0,1.536786,0.002540,0.023845,0,euc_gene
2,1,11201,11300,77.019910,16,0.443047,15,0,0.350130,12,8,2.0,113.0,1.536786,0.000022,0.000418,0,euc_gene
7,1,24301,24400,63.303077,16,0.323758,0,15,0.235239,15,16,511.0,16.0,1.536786,0.000520,0.006430,1,euc_gene
20,1,25801,25900,68.401063,16,0.283734,0,16,0.233424,9,61,373.0,8.0,1.536786,0.000165,0.002425,2,euc_gene
27,1,26601,26700,54.791427,16,0.340057,0,12,0.204524,12,17,241.0,17.0,1.536786,0.003232,0.029005,2,euc_gene
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
304620,5,15243701,15243800,61.230440,16,0.289502,3,9,0.130965,9,119,204.0,119.0,1.544736,0.000879,0.021822,0,het_te
304627,5,15259701,15259800,70.739222,16,0.413612,8,6,0.340276,6,56,25.0,56.0,1.544736,0.000105,0.004410,1,het_te
304628,5,15259801,15259900,67.105317,16,0.465839,5,4,0.305060,4,83,32.0,83.0,1.544736,0.000240,0.008304,2,het_te
304632,5,15260301,15260400,53.435277,14,0.434845,9,6,0.269258,6,31,7.0,31.0,1.544736,0.001689,0.035031,0,het_te


In [49]:
def export_to_bed(df, bed_path, score_col='X2'):
    """
    Export genomic windows with score into a BED file for IGV.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain ['chr','start','end'] and a score column (default 'X2').
    bed_path : str
        Output BED file path.
    score_col : str
        Column to export as BED score (default 'X2').
    """
    # BED requires 0-based start, 1-based end
    bed = df[['chr','start','end',score_col]].copy()
    bed['chrom'] = bed['chr'].astype(str)
    bed['chromStart'] = bed['start'].astype(int) - 1
    bed['chromEnd']   = bed['end'].astype(int)
    bed['name']  = '.'
    bed['score'] = bed[score_col].astype(float).round(6)
    bed['strand'] = '.'

    out = bed[['chrom','chromStart','chromEnd','name','score','strand']]
    out.to_csv(bed_path, sep='\t', header=False, index=False)
    print(f"Wrote {len(out)} entries to {bed_path}")


In [50]:
export_to_bed(calls_strict, "/ceph/MethDev/pbio/kay/data/bedfiles/strict_10883.bed", score_col="X2")

Wrote 10883 entries to ./data/bedfiles/strict_10883.bed


In [51]:
export_to_bed(calls_loose, "/ceph/MethDev/pbio/kay/data/bedfiles/loose_13549.bed", score_col="X2")

Wrote 13549 entries to ./data/bedfiles/loose_13549.bed


In [52]:
export_to_bed(calls_strict_base, "/ceph/MethDev/pbio/kay/data/bedfiles/strict_base_18693.bed", score_col="X2")

Wrote 18693 entries to ./data/bedfiles/strict_base_18693.bed


In [53]:
export_to_bed(calls_loose_base, "/ceph/MethDev/pbio/kay/data/bedfiles/loose_base_28558.bed", score_col="X2")

Wrote 28558 entries to ./data/bedfiles/loose_base_28558.bed
